In [31]:
import gymnasium as gym
from model import Policy
import torch
import numpy as np

In [ ]:
# IN RL 
# 1) we see the environment (observation)
# 2) Agent takes an action based on observation
# 3) Agent gets some form of reward and new state
# 4) Agent updates policy with reward.
# Rinse and Repeat

# Create the Env and reset it
env_id = "CartPole-v1"
env = gym.make(env_id)
state, info = env.reset()

# Create the policy
LR = 5e-5
GAMMA = 0.99
model = Policy()
optimizer = torch.optim.AdamW(model.parameters(), lr = LR)

# Environment HP
EPISODES = 10000
episode_steps = 25
EPISODE_SCORES = [] # Total sum of rewards

# Reset the environment.
observation, info = env.reset()

for eps in range(EPISODES):
    
    _eps_log_probs = []
    _eps_rewards = []
    observation, info = env.reset()

    for time_step in range(episode_steps):
        # Policy makes the decision.
        action, log_probs = model.act(state)

        state, reward, terminated, truncated, info = env.step(action)

        # Always add rewards before game ends.
        _eps_rewards.append(reward)
        _eps_log_probs.append(log_probs)
        
        # Game ended earlier
        if terminated or truncated:
            break
       

    EPISODE_SCORES.append(np.sum(_eps_rewards))
    # let's ignore reward to go policy for now.
    # Now to calculate policy gradient.

    # Reorders this into
    #r_t, r_t-1, r_t-2, ... , r0
    returns = []
    for r in _eps_rewards[::-1]:
        disc_return = returns[-1] if len(returns) > 0 else 0
        # print(disc_return)
        returns.append(GAMMA * disc_return + r)

    
    assert len(returns) == len(_eps_log_probs)
    eps = 1e-7
    returns = np.array(returns) 
    # Standardization (subtract by mean and divide by std)
    returns = (returns - np.mean(returns)) / (np.std(returns) + eps)
    
    log_loss_rewards = []
    for reward_to_go, log_probs in zip(returns, _eps_log_probs[::-1]):
        log_loss_rewards.append(-torch.tensor(reward_to_go) * log_probs)

    #create one tensor and sum it 
    policy_loss = torch.cat(log_loss_rewards).sum() 
    
    
    optimizer.zero_grad()
    policy_loss.backward() # compute_gradients
    optimizer.step() # update weights

    # Scores are going up!!!
    if eps % 100 == 0:
        print(f"Episode mean score: {np.mean(EPISODE_SCORES)}") 

# Copying Eval and Video Function

In [ ]:
import imageio

def evaluate_agent(env, max_steps, n_eval_episodes, policy):
    """
    Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
    :param env: The evaluation environment
    :param n_eval_episodes: Number of episode to evaluate the agent
    :param policy: The Reinforce agent
    """
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state, info = env.reset()
        done = False
        total_rewards_ep = 0

        for _ in range(max_steps):
            action, _ = policy.act(state)
            new_state, reward, terminated, truncated, info = env.step(action)
            total_rewards_ep += reward
            done = terminated or truncated

            if done:
                break
            state = new_state
        episode_rewards.append(total_rewards_ep)
    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)

    return mean_reward, std_reward

def record_video(env, policy, out_directory, fps=30):
    """
    Generate a replay video of the agent.

    Gymnasium expects the environment to be created with ``render_mode="rgb_array"``
    instead of passing ``mode`` to ``render()``.
    """
    render_env = gym.make(env.spec.id, render_mode="rgb_array")

    images = []
    done = False
    state, info = render_env.reset()
    images.append(render_env.render())

    while not done:
        action, _ = policy.act(state)
        state, reward, terminated, truncated, info = render_env.step(action)
        done = terminated or truncated
        images.append(render_env.render())

    render_env.close()
    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)

In [ ]:
rewards, std_reward = evaluate_agent(
    env,
    max_steps = 100,
    n_eval_episodes=10,
    policy=policy 
)

print(rewards, std_reward)

In [ ]:
record_video(
    env,
    policy,
    out_directory="eval.gif",
    fps=60
)